# Reproducing the Main Paper Figures

Reproduces the three main spirals figures (**Legendre** embedding, **d10r6** arch).

| Figure | Label | What it shows |
|--------|-------|---------------|
| Fig 1 | `fig:alpha-density` | Distribution heatmaps: at, α=0, 0.5, 1 |
| Fig 2 | `fig:alpha-robustness` | Accuracy/robustness + NLL vs α |
| Fig 3 | `fig:regime-spirals` | Regime barplot: 4 regimes at ε=0.2 |

**Prerequisites**: `conda activate bm4tc` before launching.  
All HPO configs are pre-filled — skip §0 unless you want fresh HPs.

**Sections**: §0 optional HPO · §1 seed sweeps · §2 analysis · §3–5 figures

---
## §0  [Optional] Re-run HPO and Fill HPs

Skip unless you want fresh hyperparameters. The seed sweep configs are already pre-filled.  
Pre-filled HPs came from `hpo_a0`, `hpo_a05`, `hpo_a1` (nat) and `hpo` (at) experiments.

In [ ]:
HPO_CONFIGS = [
    "spirals/nat/legendre/d10r6/hpo_a0",
    "spirals/nat/legendre/d10r6/hpo_a05",
    "spirals/nat/legendre/d10r6/hpo_a1",
    "spirals/at/legendre/d10r6/hpo",
]

print("# 1. Run each HPO sweep (can run in parallel):")
for cfg in HPO_CONFIGS:
    print(f"python -m experiments.train --multirun +experiments={cfg}")

print("\n# 2. Fill best HPs into seed_sweep configs (after all HPO sweeps complete):")
print("python tools/fill_hpo.py --dataset spirals --embedding legendre --arch d10r6")

---
## §1  Run Seed Sweeps

Long-running jobs — recommended to run from a terminal with tmux.
The cell below prints the commands; copy them into separate terminal panes.

> **Run order**:
> 1. Start the three `nat` sweeps and `alpha_curve` (all independent — run in parallel panes).
> 2. After `seed_sweep_a0` finishes, run `patch_checkpoint.py` to update the checkpoint path
>    in the `at/seed_sweep` config (it loads a nat/a0 model as its starting point).
> 3. Then run the `at/seed_sweep`.

| Kind | Seeds | Used by |
|------|-------|---------|
| `nat/seed_sweep_a0` | 20 | Figs 1, 2 (endpoint), 3 |
| `nat/seed_sweep_a05` | 20 | Fig 3 |
| `nat/seed_sweep_a1` | 20 | Figs 1, 2 (endpoint), 3 |
| `at/seed_sweep` | 8 | Figs 1, 3 — run after nat |
| `nat/alpha_curve` | 10α × 5 seeds | Fig 2 |

In [ ]:
NAT_SWEEPS = [
    "spirals/nat/legendre/d10r6/seed_sweep_a0",
    "spirals/nat/legendre/d10r6/seed_sweep_a05",
    "spirals/nat/legendre/d10r6/seed_sweep_a1",
]
ALPHA_CURVE = "spirals/nat/legendre/d10r6/alpha_curve"
AT_SWEEP    = "spirals/at/legendre/d10r6/seed_sweep"

# Step 1: nat sweeps + alpha_curve (run in parallel panes)
print("# Step 1 — nat sweeps + alpha_curve (independent; run in parallel panes):")
for cfg in NAT_SWEEPS:
    print(f"python -m experiments.train --multirun +experiments={cfg}")
print(f"python -m experiments.train --multirun +experiments={ALPHA_CURVE}")

# Step 2: patch at/seed_sweep checkpoint (after seed_sweep_a0 completes)
print("\n# Step 2 — patch at/seed_sweep checkpoint (after seed_sweep_a0 finishes):")
print("# Replace DDMM with the actual date suffix of your seed_sweep_a0 output dir.")
print("python tools/patch_checkpoint.py outputs/spirals/nat/legendre/d10r6/seed_sweep_a0_DDMM")

# Step 3: at sweep (after Steps 1 + 2)
print(f"\n# Step 3 — at sweep (after Steps 1 and 2 are done):")
print(f"python -m experiments.train --multirun +experiments={AT_SWEEP}")

---
## §2  Post-hoc Analysis

Fill in the dated output directories below, then run the printed commands from a terminal.

> **Distribution plots (Fig 1)**: Pass `--viz` when analysing the four seed sweeps — this
> generates `decision_boundary.png` + `best_joint.png` needed by §3. Do **not** pass `--viz`
> for `alpha_curve` (distributions there are not used for the main figures).

In [ ]:
# Fill in dated output directories after training. Format: {kind}_{DDMM}
SWEEP_DIRS = {
    "seed_sweep_a0":  "outputs/spirals/nat/legendre/d10r6/seed_sweep_a0_DDMM",
    "seed_sweep_a05": "outputs/spirals/nat/legendre/d10r6/seed_sweep_a05_DDMM",
    "seed_sweep_a1":  "outputs/spirals/nat/legendre/d10r6/seed_sweep_a1_DDMM",
    "at_seed_sweep":  "outputs/spirals/at/legendre/d10r6/seed_sweep_DDMM",
    "alpha_curve":    "outputs/spirals/nat/legendre/d10r6/alpha_curve_DDMM",
}
VIZ_SWEEPS = {"seed_sweep_a0", "seed_sweep_a05", "seed_sweep_a1", "at_seed_sweep"}

print("# Analysis commands (run from project root):")
for name, sweep_dir in SWEEP_DIRS.items():
    viz = " --viz" if name in VIZ_SWEEPS else ""
    print(f"python analysis/sweep.py {sweep_dir}{viz}")

---
## §3  Figure 1 — Distribution Panel (`fig:alpha-density`)

Assembles `decision_boundary.png` + `best_joint.png` from the four seed sweep analysis outputs
into a 2×4 grid (columns: at, α=0, α=0.5, α=1). Output: `figures/dists_with_adv.png`.

**Requires**: §2 completed with `COMPUTE_DISTRIBUTIONS = True` for the four seed sweeps.

In [ ]:
# TODO (Phase 4): Load decision_boundary.png + best_joint.png from 4 analysis output dirs
# and assemble a 2x4 matplotlib grid. Column order: [at, a0, a05, a1].
# Analysis dirs: analysis/outputs/spirals/{nat|at}/legendre/d10r6/{kind}_{DDMM}/
# Output: figures/dists_with_adv.png

---
## §4  Figure 2 — Alpha-Curve Line Plot (`fig:alpha-robustness`)

Accuracy/robustness + NLL vs α. Based on `analysis/visualize/alpha_curve_plots.py`.
The α=0 and α=1 endpoints are overridden with 20-seed data from `seed_sweep_a0`/`seed_sweep_a1`
(the `alpha_curve` multirun has only 5 seeds at those endpoints).
Output: `figures/spirals/alpha/alpha_curve.png`.

In [ ]:
# TODO (Phase 4): Adapt analysis/visualize/alpha_curve_plots.py.
# Override alpha=0 and alpha=1 endpoints with data from 20-seed seed_sweep_a0/a1 CSVs.
# (The alpha_curve multirun uses only 5 seeds at those endpoints.)
# Output: figures/spirals/alpha/alpha_curve.png

---
## §5  Figure 3 — Regime Barplot (`fig:regime-spirals`)

Barplot of clean accuracy, robustness, accepted fraction, and purified accuracy for 4 regimes
at ε=0.2. Adapted from `reference/toy_analysis.py` (`cell14_metric_barplots`).
Output: `figures/spirals/regime/metric_eps02.pdf`.

In [ ]:
# TODO (Phase 4): Adapt reference/toy_analysis.py cell14_metric_barplots() for new nat/at paths.
# Regimes: at/seed_sweep, nat/seed_sweep_a0, nat/seed_sweep_a05, nat/seed_sweep_a1
# CSV cols: eval/test/rob/0.2, eval/uq_purify_acc/0.2/0.2,
#           eval/gibbs_purify_acc/0.2/1, eval/uq_det_err_passed/10pct/0.2
# Output: figures/spirals/regime/metric_eps02.pdf